## Python 变量与基本数据类型：从“值”到“解释方式”

### 学习目标
- 理解：同一段二进制数据，在不同解释规则下会得到不同含义。
- 会定义并使用三类简单数据：整数 `int`、浮点数 `float`、字符 `char`。
- 知道：Python 没有单独的 `char` 类型，单个字符通常用长度为 1 的 `str` 表示。
- 能说清楚：`int` 的 30 bit digit、`float` 的 IEEE 754 表示、字符的 Unicode/UTF-8 表示各自解决什么问题。
- 会完成常见类型转换：`int()`、`float()`、`str()`、`ord()`、`chr()`，并知道哪些输入格式会失败。


## 变量：建模世界的基本元素
在上一节“商品总价”例子里，我们已经见过一个核心思想：程序其实是在用代码描述现实世界。
比如，商品单价、商品数量、总价，都是现实中的属性；在 Python 里，它们就会被表示成变量，如 `price`、`count`、`total`。

但变量并不是“凭空存在”的。无论是数字还是字符，最终都要以二进制（0 和 1）的形式存进计算机。
这就引出本节的关键问题：同样是二进制，Python 是如何区分它们到底是整数、浮点数还是字符的？答案就是数据类型。

## 数据类型：内容与类型共同决定了变量的含义

把二进制数据想象成纸上写着的同一串字符：`Gift`。字符本身没变，但解释它的“规则”一变，含义就会变：
- 贴上 `英语` 标签：`Gift` 常表示“礼物”；
- 贴上 `德语` 标签：`Gift` 表示“毒药”。

Python 里的数据类型也是同样道理：同样是底层二进制数，例如内存中连续 4 个字节写成 `41 42 43 00`，解释规则不同，得到的意义和可执行操作也不同。
- 贴上 `int`：按整数规则处理，如果把它看作 4 字节小端整数，结果是 `4407873`，可以做加减乘除等数值运算；
- 贴上 `str`：按文本规则处理，`41 42 43` 分别对应字符 `A`、`B`、`C`，最后的 `00` 是空字节；它更适合做拼接、切片、查找等文本操作。

In [1]:
# 同一段底层字节：41 42 43 00
stored_bytes = bytes.fromhex("41 42 43 00")

# 解释方式 1：贴上 int 标签，按 4 字节小端整数读取
int_view = int.from_bytes(stored_bytes, byteorder="little")

# 解释方式 2：贴上 str 标签，按 ASCII 文本读取前三个字节
# 最后的 00 是空字节，不作为普通可见字符展示
str_view = stored_bytes[:3].decode("ascii")
empty_byte = stored_bytes[3]

print("底层字节:", stored_bytes.hex(" "))
print("按 int 解释:", int_view)
print("按 str 解释:", str_view)
print("最后一个字节:", empty_byte)


底层字节: 41 42 43 00
按 int 解释: 4407873
按 str 解释: ABC
最后一个字节: 0


> 注：以上示例是为了简化理解，直接建立了“底层字节”和“解释结果”之间的对应关系。这种理解方式更接近 C 语言中的数据存储模型；在 Python 中，变量实际指向的是对象，对象内部还包含类型信息、引用计数、长度等额外结构。这里先不详细展开，重点把握“同一段二进制可以被不同规则解释”这一核心思想。

## 整数

### 数据定义与范围

整数（`int`）用来表示没有小数部分的数，例如人数、次数、编号、像素坐标等。

在 Python 中，`int` 不固定为 32 位或 64 位。只要内存足够，它可以表示非常大的整数。整数可以是正数、负数，也可以是 0。

为了支持“大整数”，常见的 64 位 CPython 会把整数拆成若干个内部存储块（digit）。每个 digit 使用 32 bit 的空间，但其中有效保存整数内容的是 30 bit，剩下的 bit 主要给底层计算留余量。

因此，`2**30 - 1` 刚好需要 30 bit，一个 digit 可以放下；`2**30` 需要 31 bit，就要增加一个 digit。下面用 `bit_length()` 观察需要多少二进制位，再用 `sys.getsizeof()` 观察对象大小何时增加。

In [2]:
import sys

base_size = sys.getsizeof(42)
print("小整数对象的大小:", base_size, "bytes")
print()

value = 42
size = sys.getsizeof(value)
print("小整数")
print("  value:", value)
print("  bit_length:", value.bit_length())
print("  size:", size, "bytes")
print("  相对小整数新增的 digit 数:", (size - base_size) // 4)
print()

value = 2**30 - 1
size = sys.getsizeof(value)
print("2**30 - 1")
print("  value:", value)
print("  bit_length:", value.bit_length())
print("  size:", size, "bytes")
print("  相对小整数新增的 digit 数:", (size - base_size) // 4)
print()

value = 2**30
size = sys.getsizeof(value)
print("2**30")
print("  value:", value)
print("  bit_length:", value.bit_length())
print("  size:", size, "bytes")
print("  相对小整数新增的 digit 数:", (size - base_size) // 4)
print()

value = 2**60 - 1
size = sys.getsizeof(value)
print("2**60 - 1")
print("  value:", value)
print("  bit_length:", value.bit_length())
print("  size:", size, "bytes")
print("  相对小整数新增的 digit 数:", (size - base_size) // 4)
print()

value = 2**60
size = sys.getsizeof(value)
print("2**60")
print("  value:", value)
print("  bit_length:", value.bit_length())
print("  size:", size, "bytes")
print("  相对小整数新增的 digit 数:", (size - base_size) // 4)

小整数对象的大小: 28 bytes

小整数
  value: 42
  bit_length: 6
  size: 28 bytes
  相对小整数新增的 digit 数: 0

2**30 - 1
  value: 1073741823
  bit_length: 30
  size: 28 bytes
  相对小整数新增的 digit 数: 0

2**30
  value: 1073741824
  bit_length: 31
  size: 32 bytes
  相对小整数新增的 digit 数: 1

2**60 - 1
  value: 1152921504606846975
  bit_length: 60
  size: 32 bytes
  相对小整数新增的 digit 数: 1

2**60
  value: 1152921504606846976
  bit_length: 61
  size: 36 bytes
  相对小整数新增的 digit 数: 2


### 存储与表示规则

下面这部分是 CPython 的实现细节，只需要简单理解。

在常见的 64 位 CPython 中，一个整数对象前 24 个字节主要是对象头，包括引用计数、类型指针，以及整数 digit 的数量和符号信息。从第 25 个字节开始，才是实际保存整数内容的 digit。

每个 digit 占 4 个字节，也就是 32 bit，但有效保存整数内容的是低 30 bit。整数变大时，Python 会自动增加新的 digit。

In [3]:
import ctypes

value = 2**30 - 1
address = id(value)
raw = ctypes.string_at(address, 32)
header = raw[:24]
first_digit = raw[24:28]
second_digit = raw[28:32]

print("2**30 - 1")
print("  对象地址:", hex(address))
print("  前 24 字节对象头:", header.hex(" "))
print("  第 1 个 digit:", first_digit.hex(" "), "=", int.from_bytes(first_digit, "little"))
print("  第 2 个 digit:", second_digit.hex(" "), "=", int.from_bytes(second_digit, "little"))

value = 2**30
address = id(value)
raw = ctypes.string_at(address, 32)
header = raw[:24]
first_digit = raw[24:28]
second_digit = raw[28:32]

print("2**30")
print("  对象地址:", hex(address))
print("  前 24 字节对象头:", header.hex(" "))
print("  第 1 个 digit:", first_digit.hex(" "), "=", int.from_bytes(first_digit, "little"))
print("  第 2 个 digit:", second_digit.hex(" "), "=", int.from_bytes(second_digit, "little"))

2**30 - 1
  对象地址: 0x1054a5870
  前 24 字节对象头: 01 00 00 00 00 00 00 00 f0 1e e1 00 01 00 00 00 08 00 00 00 00 00 00 00
  第 1 个 digit: ff ff ff 3f = 1073741823
  第 2 个 digit: 00 00 00 00 = 0
2**30
  对象地址: 0x1054a56b0
  前 24 字节对象头: 01 00 00 00 00 00 00 00 f0 1e e1 00 01 00 00 00 10 00 00 00 00 00 00 00
  第 1 个 digit: 00 00 00 00 = 0
  第 2 个 digit: 01 00 00 00 = 1


### 支持的运算

设 `a = 7`，`b = 2`，整数常见运算如下：

| 运算符 | 名称 | 示例 | 结果 | 说明 |
|---|---|---|---|---|
| `+` | 加法 | `a + b` | `9` | 两个整数相加 |
| `-` | 减法 | `a - b` | `5` | 两个整数相减 |
| `*` | 乘法 | `a * b` | `14` | 两个整数相乘 |
| `/` | 普通除法 | `a / b` | `3.5` | 结果总是 `float` |
| `//` | 整除 | `a // b` | `3` | 只保留整数商 |
| `%` | 取余 | `a % b` | `1` | 得到除法余数 |
| `**` | 幂运算 | `a ** b` | `49` | 表示 a 的 b 次方 |


In [4]:
a = 7
b = 2

print("a + b =", a + b)
print("a - b =", a - b)
print("a * b =", a * b)
print("a / b =", a / b, "type:", type(a / b))
print("a // b =", a // b)
print("a % b =", a % b)
print("a ** b =", a ** b)

a + b = 9
a - b = 5
a * b = 14
a / b = 3.5 type: <class 'float'>
a // b = 3
a % b = 1
a ** b = 49


## 浮点数

### 数据定义与范围

浮点数（`float`）用来表示带小数部分的数，例如价格、长度、概率、模型损失值等。

Python 的 `float` 通常使用 IEEE 754 双精度浮点数（double precision），占 64 bit。它能表示非常大或非常小的数，但精度是有限的。

In [5]:
import sys

price = 12.5
probability = 0.95
scientific = 1.2e3

print("price =", price, "type:", type(price))
print("probability =", probability, "type:", type(probability))
print("scientific =", scientific, "type:", type(scientific))
print("float 最大值:", sys.float_info.max)
print("float 最小正规格正数:", sys.float_info.min)
print("float 精度位数:", sys.float_info.mant_dig)

price = 12.5 type: <class 'float'>
probability = 0.95 type: <class 'float'>
scientific = 1200.0 type: <class 'float'>
float 最大值: 1.7976931348623157e+308
float 最小正规格正数: 2.2250738585072014e-308
float 精度位数: 53


### 存储与表示规则

浮点数不是直接存“十进制小数”。它更像科学计数法：用符号位、指数和尾数共同表示一个数。

以常见的 IEEE 754 双精度为例，64 bit 大致分成三部分：

| 部分 | 位数 | 作用 |
|---|---:|---|
| 符号位 sign | 1 bit | 表示正数或负数 |
| 指数 exponent | 11 bit | 表示数量级 |
| 尾数 fraction | 52 bit | 表示有效数字 |

因此，浮点数能表示很大的范围，但不是所有十进制小数都能被精确表示。

In [6]:
import struct

x = 12.5

# 使用大端顺序展示 8 个字节，便于从左到右观察 sign/exponent/fraction
raw = struct.pack(">d", x)
bits = int.from_bytes(raw, "big")

sign = (bits >> 63) & 1 # 1 位符号
exponent = (bits >> 52) & 0x7ff # 11 位指数
fraction = bits & ((1 << 52) - 1) # 52 位尾数

print("x =", x)
print("IEEE 754 双精度字节:", raw.hex(" "))
print("符号位 sign:", sign)
print("指数 exponent:", exponent)
print("尾数 fraction:", hex(fraction))
print("12.5 的二进制可以写成: 1.5625 * 2**3")

# 数学公式：value = (-1)**sign * (1 + fraction / 2**52) * 2**(exponent - 1023)
bias = 1023
real_exponent = exponent - bias
mantissa = 1 + fraction / 2**52
value_formula = f"(-1)**{sign} * {mantissa} * 2**{real_exponent}"
restored = (-1) ** sign * mantissa * 2**real_exponent

print("偏置 bias:", bias)
print("真实指数 exponent - bias:", real_exponent)
print("有效尾数 1 + fraction / 2**52:", mantissa)
print("代入公式:", value_formula)
print("公式还原: (-1)**sign * mantissa * 2**real_exponent =", restored)

x = 12.5
IEEE 754 双精度字节: 40 29 00 00 00 00 00 00
符号位 sign: 0
指数 exponent: 1026
尾数 fraction: 0x9000000000000
12.5 的二进制可以写成: 1.5625 * 2**3
偏置 bias: 1023
真实指数 exponent - bias: 3
有效尾数 1 + fraction / 2**52: 1.5625
代入公式: (-1)**0 * 1.5625 * 2**3
公式还原: (-1)**sign * mantissa * 2**real_exponent = 12.5


### 支持的运算

设 `a = 7.5`，`b = 2.0`，浮点数常见运算如下：

| 运算符 | 名称 | 示例 | 结果 | 说明 |
|---|---|---|---|---|
| `+` | 加法 | `a + b` | `9.5` | 两个浮点数相加 |
| `-` | 减法 | `a - b` | `5.5` | 两个浮点数相减 |
| `*` | 乘法 | `a * b` | `15.0` | 两个浮点数相乘 |
| `/` | 除法 | `a / b` | `3.75` | 保留小数结果 |
| `**` | 幂运算 | `a ** b` | `56.25` | 表示 a 的 b 次方 |

需要特别注意精度问题：像 `0.1`、`0.2` 这样的十进制小数，在二进制浮点数中通常不能被精确表示。

In [7]:
a = 7.5
b = 2.0

print("a + b =", a + b)
print("a - b =", a - b)
print("a * b =", a * b)
print("a / b =", a / b)
print("a ** b =", a ** b)

print("0.1 + 0.2 =", 0.1 + 0.2)
print("round(0.1 + 0.2, 1) =", round(0.1 + 0.2, 1))

a + b = 9.5
a - b = 5.5
a * b = 15.0
a / b = 3.75
a ** b = 56.25
0.1 + 0.2 = 0.30000000000000004
round(0.1 + 0.2, 1) = 0.3


## 字符 `char`：Python 中用长度为 1 的 `str` 表示

#### 数据定义与范围

很多语言有专门的 `char` 类型，例如 C 语言中的字符。Python 更简单：没有单独的 `char` 类型，单个字符通常用长度为 1 的字符串（`str`）表示。

Python 字符基于 Unicode。Unicode 给世界上大量字符分配统一编号，例如 `A` 的编号是 `0x41`，`中` 的编号是 `0x4e2d`。

In [8]:
ch = "A"
print("英文字母")
print("  char:", ch)
print("  type:", type(ch))
print("  len:", len(ch))
print("  Unicode 编号:", ord(ch))
print("  十六进制编号:", hex(ord(ch)))

ch = "中"
print("中文字符")
print("  char:", ch)
print("  type:", type(ch))
print("  len:", len(ch))
print("  Unicode 编号:", ord(ch))
print("  十六进制编号:", hex(ord(ch)))

ch = "😊"
print("表情字符")
print("  char:", ch)
print("  type:", type(ch))
print("  len:", len(ch))
print("  Unicode 编号:", ord(ch))
print("  十六进制编号:", hex(ord(ch)))

print("Unicode 最大码点:", hex(0x10ffff))
print("chr(65) =", chr(65))
print("chr(0x4e2d) =", chr(0x4e2d))

英文字母
  char: A
  type: <class 'str'>
  len: 1
  Unicode 编号: 65
  十六进制编号: 0x41
中文字符
  char: 中
  type: <class 'str'>
  len: 1
  Unicode 编号: 20013
  十六进制编号: 0x4e2d
表情字符
  char: 😊
  type: <class 'str'>
  len: 1
  Unicode 编号: 128522
  十六进制编号: 0x1f60a
Unicode 最大码点: 0x10ffff
chr(65) = A
chr(0x4e2d) = 中


#### 存储与表示规则

字符在 Python 中先被看作 Unicode 字符。真正写入文件、网络传输或转成字节时，还需要选择编码方式。最常见的是 UTF-8。

UTF-8 是变长编码：英文字符通常占 1 个字节，常见中文字符通常占 3 个字节，一些表情字符通常占 4 个字节。

In [9]:
ch = "A"
utf8_bytes = ch.encode("utf-8")
print("字符:", ch)
print("  Unicode 编号:", hex(ord(ch)))
print("  UTF-8 字节:", utf8_bytes.hex(" "))
print("  UTF-8 字节数:", len(utf8_bytes))

ch = "中"
utf8_bytes = ch.encode("utf-8")
print("字符:", ch)
print("  Unicode 编号:", hex(ord(ch)))
print("  UTF-8 字节:", utf8_bytes.hex(" "))
print("  UTF-8 字节数:", len(utf8_bytes))

ch = "😊"
utf8_bytes = ch.encode("utf-8")
print("字符:", ch)
print("  Unicode 编号:", hex(ord(ch)))
print("  UTF-8 字节:", utf8_bytes.hex(" "))
print("  UTF-8 字节数:", len(utf8_bytes))

字符: A
  Unicode 编号: 0x41
  UTF-8 字节: 41
  UTF-8 字节数: 1
字符: 中
  Unicode 编号: 0x4e2d
  UTF-8 字节: e4 b8 ad
  UTF-8 字节数: 3
字符: 😊
  Unicode 编号: 0x1f60a
  UTF-8 字节: f0 9f 98 8a
  UTF-8 字节数: 4


#### 支持的运算

因为 Python 中的字符本质上是长度为 1 的 `str`，所以它支持字符串的基本操作。这里只列出和“单个字符”最相关的操作。

| 操作 | 示例 | 结果 | 说明 |
|---|---|---|---|
| 判断相等 | `ch == "A"` | `True` | 判断是否是同一个字符 |
| 编号转换 | `ord(ch)` | `65` | 字符转 Unicode 编号 |
| 编号还原 | `chr(66)` | `"B"` | Unicode 编号转字符 |
| 拼接 | `ch + "BC"` | `"ABC"` | 字符可以和字符串拼接 |
| 重复 | `ch * 3` | `"AAA"` | 重复生成字符串 |
| 比较 | `ch < "B"` | `True` | 按 Unicode 编号顺序比较 |

In [10]:
ch = "A"

print("ch == 'A' ->", ch == "A")
print("ord(ch) ->", ord(ch))
print("chr(66) ->", chr(66))
print("ch + 'BC' ->", ch + "BC")
print("ch * 3 ->", ch * 3)
print("ch < 'B' ->", ch < "B")

ch == 'A' -> True
ord(ch) -> 65
chr(66) -> B
ch + 'BC' -> ABC
ch * 3 -> AAA
ch < 'B' -> True


In [11]:
# 先看一个合法转换
print("int('123') =", int("123"))

# 下面两种写法会报 ValueError，所以这里先不直接运行
print("int('12.5') 不能直接运行：'12.5' 是小数字符串")
print("int('abc') 不能直接运行：'abc' 不是数字")

int('123') = 123
int('12.5') 不能直接运行：'12.5' 是小数字符串
int('abc') 不能直接运行：'abc' 不是数字


### 类型转换

类型转换就是把一个值从一种解释规则转换成另一种解释规则。初学阶段最常见的是：把输入得到的文本转成数字，或者把数字转成文本用于输出。

| 转换函数 | 作用 | 有效输入 | 格式要求 | 示例 |
|---|---|---|---|---|
| `int(x)` | 转成整数 | 整数字符串、整数、可截断的浮点数 | 字符串必须是合法整数，如 `"12"`、`"-3"`；`"12.5"` 不能直接转成 `int` | `int("12") -> 12` |
| `float(x)` | 转成浮点数 | 数字字符串、整数、浮点数 | 字符串必须是合法小数或科学计数法，如 `"3.5"`、`"1e-3"` | `float("3.5") -> 3.5` |
| `str(x)` | 转成字符串 | 几乎任意 Python 对象 | 没有严格格式要求，会生成该对象的文本形式 | `str(42.0) -> "42.0"` |
| `ord(ch)` | 字符转 Unicode 编号 | 单个字符 | 输入必须是长度为 1 的字符串 | `ord("A") -> 65` |
| `chr(n)` | Unicode 编号转字符 | 整数 | 输入必须是合法 Unicode 码点，范围是 `0` 到 `0x10ffff` | `chr(65) -> "A"` |

注意：转换不是总能成功。比如 `int("abc")` 会报错，因为 `"abc"` 不是合法整数。

In [12]:
raw_count = "12"
raw_price = "3.5"

count = int(raw_count)
price = float(raw_price)
total = count * price

letter = "A"
code = ord(letter)

print("int('12') ->", count, type(count))
print("float('3.5') ->", price, type(price))
print("str(total) ->", str(total), type(str(total)))
print("ord('A') ->", code)
print("chr(65) ->", chr(65))

int('12') -> 12 <class 'int'>
float('3.5') -> 3.5 <class 'float'>
str(total) -> 42.0 <class 'str'>
ord('A') -> 65
chr(65) -> A


## 习题

下面的题目用于检查你是否真正理解了“类型就是解释规则”。代码题采用填空形式，填完后运行单元格，若没有报错并输出“验证通过”，说明答案正确。

### 1. 概念判断：同一段字节为什么会有不同含义？

同样是 `41 42 43 00` 这 4 个字节，为什么按 `int` 解释会得到 `4407873`，按文本解释会得到 `ABC`？请用自己的话说明“解释规则”在其中起什么作用。

<details>
<summary>参考答案</summary>

底层字节本身没有变。按 `int` 解释时，Python 把它看作小端整数；按文本解释时，Python 把 `41 42 43` 看作 ASCII 编码，对应 `A`、`B`、`C`。类型决定了同一段二进制如何被理解，以及后续能做什么操作。

</details>

### 2. 概念判断：为什么 `2**30` 会增加一个 digit？

`2**30 - 1` 和 `2**30` 只差 1，为什么在 CPython 的整数对象中，`2**30` 需要增加一个 digit？

<details>
<summary>参考答案</summary>

常见 64 位 CPython 中，一个整数 digit 有效保存 30 bit。`2**30 - 1` 的二进制刚好是 30 个有效位，一个 digit 能放下；`2**30` 需要 31 bit，超过一个 digit 的有效容量，因此需要新增一个 digit。

</details>

### 3. 概念判断：为什么 `0.1 + 0.2` 不等于刚好 `0.3`？

请结合浮点数的存储方式，解释为什么 `0.1 + 0.2` 会得到 `0.30000000000000004`。

<details>
<summary>参考答案</summary>

`float` 通常使用 IEEE 754 二进制浮点数。很多十进制小数，例如 `0.1` 和 `0.2`，不能被二进制浮点数精确表示，只能存成近似值。两个近似值相加后，结果也可能带有很小的误差。

</details>

### 4. 代码填空：同一段字节的两种解释

请补全 `answer_int` 和 `answer_text`，让同一段字节分别按整数和文本规则解释。

In [ ]:
stored_bytes = bytes.fromhex("41 42 43 00")

# 填空 1：按 4 字节小端整数解释
answer_int = ...

# 填空 2：按 ASCII 文本解释前三个字节
answer_text = ...

if answer_int is ... or answer_text is ...:
    print("请先完成填空。")
else:
    assert answer_int == 4407873
    assert answer_text == "ABC"
    print("第 4 题验证通过")

<details>
<summary>第 4 题参考答案</summary>

```python
answer_int = int.from_bytes(stored_bytes, byteorder="little")
answer_text = stored_bytes[:3].decode("ascii")
```

</details>

### 5. 代码填空：把输入文本转成可计算的数字

`raw_count` 和 `raw_price` 都是字符串。请把它们转换成合适的数字类型，并计算总价。

In [ ]:
raw_count = "18"
raw_price = "2.5"

# 填空 1：数量应该转成整数
count = ...

# 填空 2：价格应该转成浮点数
price = ...

# 填空 3：计算总价
total = ...

if count is ... or price is ... or total is ...:
    print("请先完成填空。")
else:
    assert count == 18
    assert price == 2.5
    assert total == 45.0
    assert str(total) == "45.0"
    print("第 5 题验证通过")

<details>
<summary>第 5 题参考答案</summary>

```python
count = int(raw_count)
price = float(raw_price)
total = count * price
```

</details>

### 6. 代码填空：字符、Unicode 与 UTF-8

请补全代码，观察字符 `中` 的 Unicode 编号和 UTF-8 字节表示。

In [ ]:
ch = "中"

# 填空 1：得到 Unicode 编号
unicode_code = ...

# 填空 2：得到 UTF-8 字节
utf8_bytes = ...

if unicode_code is ... or utf8_bytes is ...:
    print("请先完成填空。")
else:
    assert unicode_code == 0x4e2d
    assert utf8_bytes.hex(" ") == "e4 b8 ad"
    assert chr(unicode_code) == ch
    print("第 6 题验证通过")

<details>
<summary>第 6 题参考答案</summary>

```python
unicode_code = ord(ch)
utf8_bytes = ch.encode("utf-8")
```

对字符 `中` 来说：

- `ord("中")` 得到十进制编号 `20013`，也就是十六进制 `0x4e2d`。
- `"中".encode("utf-8")` 得到 3 个字节：`e4 b8 ad`。
- `chr(0x4e2d)` 可以把 Unicode 编号还原成字符 `中`。

</details>